In [ ]:
!pip install fastapi uvicorn pyngrok transformers accelerate bitsandbytes nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00


In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.3",
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import traceback

app = FastAPI()


class PromptRequest(BaseModel):
    prompt: str


@app.get("/")
def home():
    return {
        "message": "Mistral LLM Server Running"
    }


@app.post("/generate")
def generate(req: PromptRequest):

    try:

        print("=" * 50)
        print("PROMPT RECEIVED")
        print(req.prompt)

        output = generator(
    req.prompt,
    max_new_tokens=350,
    do_sample=True,
    temperature=0.3,
    return_full_text=False
)

        print("=" * 50)
        print("OUTPUT")
        print(output)

        return {
            "response":
            output[0]["generated_text"]
        }

    except Exception as e:

        print("=" * 50)
        print("FULL ERROR")
        traceback.print_exc()

        return {
            "error":
            str(e)
        }

In [ ]:
import nest_asyncio
import threading
import uvicorn

nest_asyncio.apply()

def run():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

threading.Thread(
    target=run,
    daemon=True
).start()

print("FASTAPI STARTED")

FASTAPI STARTED


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package cloudflared.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.6.0) ...
Setting up cloudflared (2026.6.0) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!cloudflared tunnel --url http://localhost:8000

2026-06-09T12:15:14Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-09T12:15:14Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-09T12:15:17Z INF +--------------------------------------------------------------------------------------------+
2026-06-09T12:15:17Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-09T12:15:17Z INF |  https://bizarre-advertising-reflects-banana.trycloudf

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


PROMPT RECEIVED

You are generating a complaint for a Government Grievance Portal.

Citizen Name: Citizen
Location: Near Balapur Bus Stop
Address: Hyderabad
Department: Sanitation Department

Complaint:
Garbage has not been collected for several days causing bad smell in the area.

Requirements:

1. Write a complete formal complaint letter.
2. Include a subject line.
3. Explain the issue in detail.
4. Mention public inconvenience and safety concerns.
5. Request immediate action.
6. End with:

Yours faithfully,
Citizen

7. Do not use placeholders.
8. Do not generate examples.
9. Do not generate notes.
10. Return only the complaint.



[transformers] Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


OUTPUT
[{'generated_text': '\nSubject: Urgent: Garbage Collection Issue Near Balapur Bus Stop, Hyderabad\n\nDear Sanitation Department,\n\nI am writing this letter to bring to your attention a pressing issue regarding the sanitation services in my locality near Balapur Bus Stop, Hyderabad.\n\nFor several days now, I have noticed that the garbage has not been collected, resulting in a foul odor that is causing significant inconvenience to the residents and passersby. This not only affects the quality of life but also poses a potential health and safety risk, especially during these times when maintaining cleanliness and hygiene is of utmost importance.\n\nI kindly request that you look into this matter immediately and arrange for the garbage to be collected without delay. I believe that prompt action will help restore the cleanliness and hygiene of our locality, ensuring the well-being of all its inhabitants.\n\nI look forward to your prompt response and action in this regard.\n\nYours 

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT RECEIVED

You are generating a complaint for a Government Grievance Portal.

Citizen Name: Citizen
Location: Near Charminar Main Road
Address: Hyderabad
Department: Electricity Department

Complaint:
Street lights are not working during night causing safety concerns.

Requirements:

1. Write a complete formal complaint letter.
2. Include a subject line.
3. Explain the issue in detail.
4. Mention public inconvenience and safety concerns.
5. Request immediate action.
6. End with:

Yours faithfully,
Citizen

7. Do not use placeholders.
8. Do not generate examples.
9. Do not generate notes.
10. Return only the complaint.

OUTPUT
[{'generated_text': '\nSubject: Complaint Regarding Non-Functional Street Lights on Charminar Main Road, Hyderabad\n\nDear Sir/Madam,\n\nI am writing this letter to bring to your attention a matter of public safety and inconvenience that has been persisting for quite some time now. The street lights on Charminar Main Road in Hyderabad have been non-functiona

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT RECEIVED

You are generating a complaint for a Government Grievance Portal.

Citizen Name: Citizen
Location: Near center of town 
Address: Hyderabad
Department: Electricity Department

Complaint:
there is a electricity problem in the center of the town from 4 days

Requirements:

1. Write a complete formal complaint letter.
2. Include a subject line.
3. Explain the issue in detail.
4. Mention public inconvenience and safety concerns.
5. Request immediate action.
6. End with:

Yours faithfully,
Citizen

7. Do not use placeholders.
8. Do not generate examples.
9. Do not generate notes.
10. Return only the complaint.

OUTPUT
[{'generated_text': '\nSubject: Urgent Complaint Regarding Persistent Electricity Problem in the Center of Town, Hyderabad\n\nDear Sir/Madam,\n\nI hope this letter finds you in good health and high spirits. I am writing to bring to your attention a matter of great concern that has been causing significant public inconvenience and safety risks in the center of o